# FUSE Stage 2 — Hunyuan3D-2.1 intact-shape prior

This notebook generates an **untextured intact-shape hypothesis** from a reference image. It is intentionally geometry-only:

```text
intact reference image
→ optional background removal
→ Hunyuan3D-Shape 2.1
→ seed candidates
→ topology report + interactive previews
→ manual candidate selection
→ intact_prior.glb for Kaolin alignment
```

The output is **inferred, normalized and non-metric**. It cannot replace the measured VGGT geometry. The next container (fuse-kaolin) estimates a similarity transform from the common surviving surfaces.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Any, Dict

import numpy as np
import plotly.graph_objects as go
import torch
import trimesh
from IPython.display import display
from PIL import Image

from hy3dshape.pipelines import Hunyuan3DDiTFlowMatchingPipeline
from hy3dshape.rembg import BackgroundRemover

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print("GPU:", properties.name)
    print(f"VRAM: {properties.total_memory / 2**30:.1f} GiB")


In [ ]:
# -------------------------
# USER CONFIGURATION
# -------------------------

MODEL_ID = "tencent/Hunyuan3D-2.1"
MODEL_SUBFOLDER = "hunyuan3d-dit-v2-1"

RUN_NAME = "run_001"
SEEDS = [144, 1024]  # Add two more seeds only after the first run succeeds.
SELECTED_SEED = SEEDS[1]  # Change this after comparing the previews.

AUTO_REMOVE_BACKGROUND = True
OVERWRITE_CANDIDATES = True
LOW_VRAM_MODE = False  # Set True only if the normal 12 GB run raises CUDA OOM.

NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 5.0
OCTREE_RESOLUTION = 384  # Smoke test: 256. Final candidate: 384.
MAX_PREVIEW_POINTS = 150_000 # 100_000

# -------------------------
# FUSE PATHS
# -------------------------

FUSE_ROOT = Path("/workspace") if Path("/workspace/data").exists() else Path.cwd().resolve()
PRIOR_ROOT = FUSE_ROOT / "data" 
INPUT_IMAGE = PRIOR_ROOT / "scenes" / "global" / "reference" / "intact_ref.png"
RUN_DIR = PRIOR_ROOT / "hunyuan_outputs" / "runs" / RUN_NAME
CANDIDATE_DIR = RUN_DIR / "candidates"
SELECTED_DIR = RUN_DIR / "selected"
PREPARED_IMAGE = RUN_DIR / "prepared_condition.png"

for folder in [INPUT_IMAGE.parent, CANDIDATE_DIR, SELECTED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Input:", INPUT_IMAGE)
print("Run directory:", RUN_DIR)
print("Seeds:", SEEDS)
print("Selected seed:", SELECTED_SEED)


## 1. Prepare the conditioning image

Use an image of the **intact** Nessie, with the complete silhouette visible (this was AI generated, since the tip has not been there for a long time). The background remover is skipped when the file already contains a useful alpha channel. 

In [ ]:
if not INPUT_IMAGE.exists():
    raise FileNotFoundError(
        f"Place the intact reference image at {INPUT_IMAGE} and rerun this cell."
    )

source_image = Image.open(INPUT_IMAGE).convert("RGBA")
alpha = np.asarray(source_image.getchannel("A"))
has_useful_alpha = bool(alpha.min() < 250 and np.count_nonzero(alpha > 20) > 0)

if AUTO_REMOVE_BACKGROUND and not has_useful_alpha:
    print("No useful alpha channel detected; running background removal on CPU...")
    background_remover = BackgroundRemover()
    condition_image = background_remover(source_image.convert("RGB")).convert("RGBA")
else:
    print("Using the existing image/alpha channel.")
    condition_image = source_image

condition_image.save(PREPARED_IMAGE)

display(source_image)
display(condition_image)
print("Prepared condition saved to:", PREPARED_IMAGE)
print("Prepared size:", condition_image.size)


## 2. Load the shape model

The first execution downloads the official shape checkpoint. Normal mode keeps the pipeline on the GPU. Low-VRAM mode loads on CPU and moves one model component at a time to the GPU; it is slower but useful if desktop processes leave too little free VRAM.


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("This FUSE notebook requires an NVIDIA CUDA GPU.")

load_device = "cpu" if LOW_VRAM_MODE else "cuda"
pipeline = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    MODEL_ID,
    subfolder=MODEL_SUBFOLDER,
    device=load_device,
    dtype=torch.float16,
)

if LOW_VRAM_MODE:
    pipeline.enable_model_cpu_offload(gpu_id=0, device="cuda")

print("Pipeline loaded. Low-VRAM mode:", LOW_VRAM_MODE)


In [ ]:
def source_commit():
    repo = Path("/opt/Hunyuan3D-2.1")

    if not (repo / ".git").exists():
        return "unknown (git metadata not included in image)"

    try:
        return subprocess.check_output(
            [
                "git",
                "-c",
                f"safe.directory={repo}",
                "-C",
                str(repo),
                "rev-parse",
                "HEAD",
            ],
            text=True,
            stderr=subprocess.STDOUT,
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError) as exc:
        print(f"Warning: could not determine Hunyuan source commit: {exc}")
        return "unknown"


def ensure_mesh(value: Any) -> trimesh.Trimesh:
    if isinstance(value, trimesh.Trimesh):
        return value
    if isinstance(value, trimesh.Scene):
        geometries = tuple(value.geometry.values())
        if not geometries:
            raise ValueError("The generated scene contains no mesh geometry.")
        return trimesh.util.concatenate(geometries)
    raise TypeError(f"Expected Trimesh or Scene, received {type(value)!r}")


def mesh_health(mesh: trimesh.Trimesh) -> Dict[str, Any]:
    components = mesh.split(only_watertight=False)
    return {
        "vertices": int(len(mesh.vertices)),
        "faces": int(len(mesh.faces)),
        "components": int(len(components)),
        "watertight": bool(mesh.is_watertight),
        "winding_consistent": bool(mesh.is_winding_consistent),
        "euler_number": int(mesh.euler_number),
        "bounds": np.asarray(mesh.bounds).tolist(),
        "extents_model_units": np.asarray(mesh.extents).tolist(),
        "volume_model_units_cubed": float(mesh.volume) if mesh.is_watertight else None,
    }


def make_preview(
    mesh: trimesh.Trimesh,
    title: str,
    html_path: Path,
    seed: int,
) -> go.Figure:
    sample_count = min(MAX_PREVIEW_POINTS, max(20_000, len(mesh.faces)))
    try:
        points, face_ids = trimesh.sample.sample_surface(
            mesh, sample_count, seed=seed
        )
    except TypeError:  # Compatibility with older Trimesh builds.
        np.random.seed(seed)
        points, face_ids = trimesh.sample.sample_surface(mesh, sample_count)
    normal_z = mesh.face_normals[face_ids, 2]

    figure = go.Figure(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker={
                "size": 1.4,
                "color": normal_z,
                "colorscale": "RdYlBu",
                "showscale": False,
                "opacity": 0.9,
            },
        )
    )
    figure.update_layout(
        title=title,
        scene={"aspectmode": "data"},
        width=950,
        height=750,
        margin={"l": 0, "r": 0, "b": 0, "t": 45},
    )
    figure.write_html(str(html_path), include_plotlyjs=True, full_html=True)
    return figure


## 3. Generate candidate meshes

Each seed is generated and saved independently. The `raw` GLB is the direct model output. The `alignment_ready` copy receives only conservative Trimesh validation/normal cleanup; the measured object is not involved at this stage. Existing candidates are reused unless `OVERWRITE_CANDIDATES=True`.


In [ ]:
candidate_records = []
candidate_figures = {}
input_sha256 = hashlib.sha256(INPUT_IMAGE.read_bytes()).hexdigest()

for seed in SEEDS:
    candidate_name = f"seed_{seed}"
    candidate_path = CANDIDATE_DIR / candidate_name
    candidate_path.mkdir(parents=True, exist_ok=True)

    raw_glb = candidate_path / "intact_prior_raw.glb"
    alignment_glb = candidate_path / "intact_prior_alignment_ready.glb"
    alignment_ply = candidate_path / "intact_prior_alignment_ready.ply"
    preview_html = candidate_path / "intact_prior_preview.html"
    report_path = candidate_path / "candidate_report.json"

    if alignment_glb.exists() and report_path.exists() and not OVERWRITE_CANDIDATES:
        print(f"[{candidate_name}] Reusing existing candidate.")
        alignment_mesh = ensure_mesh(trimesh.load(alignment_glb, force="mesh"))
        report = json.loads(report_path.read_text())
    else:
        print(f"[{candidate_name}] Generating...")
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        generator = torch.Generator(device="cuda").manual_seed(seed)
        started = time.perf_counter()

        generated = pipeline(
            image=condition_image,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            octree_resolution=OCTREE_RESOLUTION,
            generator=generator,
            output_type="trimesh",
        )[0]
        elapsed = time.perf_counter() - started

        raw_mesh = ensure_mesh(generated)
        if len(raw_mesh.vertices) == 0 or len(raw_mesh.faces) == 0:
            raise RuntimeError(f"{candidate_name} produced an empty mesh.")

        raw_mesh.export(raw_glb)
        alignment_mesh = raw_mesh.copy()
        alignment_mesh.process(validate=True)
        alignment_mesh.remove_unreferenced_vertices()
        alignment_mesh.fix_normals()
        alignment_mesh.export(alignment_glb)
        alignment_mesh.export(alignment_ply)

        report = {
            "candidate": candidate_name,
            "seed": int(seed),
            "model_id": MODEL_ID,
            "model_subfolder": MODEL_SUBFOLDER,
            "source_commit": source_commit(),
            "input_image": str(INPUT_IMAGE),
            "input_sha256": input_sha256,
            "prepared_condition": str(PREPARED_IMAGE),
            "settings": {
                "num_inference_steps": NUM_INFERENCE_STEPS,
                "guidance_scale": GUIDANCE_SCALE,
                "octree_resolution": OCTREE_RESOLUTION,
                "low_vram_mode": LOW_VRAM_MODE,
            },
            "runtime_seconds": float(elapsed),
            "peak_cuda_memory_gib": float(torch.cuda.max_memory_allocated() / 2**30),
            "raw_mesh": mesh_health(raw_mesh),
            "alignment_ready_mesh": mesh_health(alignment_mesh),
            "coordinate_status": {
                "units": "Hunyuan normalized model units; not millimetres",
                "pose": "Hunyuan canonical pose; not registered to VGGT",
                "authority": "inferred intact-shape hypothesis",
            },
            "paths": {
                "raw_glb": str(raw_glb),
                "alignment_glb": str(alignment_glb),
                "alignment_ply": str(alignment_ply),
                "preview_html": str(preview_html),
            },
        }
        report_path.write_text(json.dumps(report, indent=2))

    figure = make_preview(
        alignment_mesh,
        title=f"Hunyuan3D intact prior — {candidate_name}",
        html_path=preview_html,
        seed=seed,
    )
    candidate_figures[seed] = figure
    candidate_records.append(report)
    print(json.dumps(report["alignment_ready_mesh"], indent=2))

print(f"Prepared {len(candidate_records)} candidate(s).")


In [ ]:
# Inspect every candidate interactively before choosing SELECTED_SEED.
for seed, figure in candidate_figures.items():
    print(f"Seed {seed}")
    display(figure)


## 4. Select the FUSE prior

Choose the candidate by geometry. In this specific case, one might check:

- the body, neck and surviving tail can plausibly align with the broken VGGT reconstruction;
- the inferred missing tail continues in the correct direction and ends with the desired moderately spiky tip;
- there are no duplicate limbs/tails or catastrophic rear-view hallucinations;
- thin features are not disconnected;
- watertightness and component counts are recorded, but visual suitability comes first because the geometry container performs final repair and print QA.

Set `SELECTED_SEED` in the configuration cell, rerun that cell, then run the promotion cell below.


In [ ]:
if SELECTED_SEED not in SEEDS:
    raise ValueError(f"SELECTED_SEED={SELECTED_SEED} is not in SEEDS={SEEDS}")

source_folder = CANDIDATE_DIR / f"seed_{SELECTED_SEED}"
source_glb = source_folder / "intact_prior_alignment_ready.glb"
source_ply = source_folder / "intact_prior_alignment_ready.ply"
source_preview = source_folder / "intact_prior_preview.html"
source_report = source_folder / "candidate_report.json"

for required in [source_glb, source_ply, source_preview, source_report]:
    if not required.exists():
        raise FileNotFoundError(required)

final_glb = SELECTED_DIR / "intact_prior.glb"
final_ply = SELECTED_DIR / "intact_prior.ply"
final_preview = SELECTED_DIR / "intact_prior_preview.html"
final_manifest = SELECTED_DIR / "prior_manifest.json"

shutil.copy2(source_glb, final_glb)
shutil.copy2(source_ply, final_ply)
shutil.copy2(source_preview, final_preview)
selected_report = json.loads(source_report.read_text())

manifest = {
    "stage": "02_prior",
    "run_name": RUN_NAME,
    "selected_seed": int(SELECTED_SEED),
    "model_id": MODEL_ID,
    "source_commit": source_commit(),
    "input_image": str(INPUT_IMAGE),
    "input_sha256": input_sha256,
    "main_output": str(final_glb),
    "coordinate_status": selected_report["coordinate_status"],
    "mesh_health": selected_report["alignment_ready_mesh"],
    "next_stage_contract": {
        "fixed_geometry": "VGGT broken object",
        "moving_geometry": "this intact prior",
        "transform": "Sim(3): scale, rotation, translation",
        "exclude_from_alignment": ["inferred missing tip", "measured fracture face"],
    },
}
final_manifest.write_text(json.dumps(manifest, indent=2))

print("Promoted seed:", SELECTED_SEED)
print("Kaolin handoff:", final_glb)
print("Manifest:", final_manifest)


In [ ]:
required_outputs = [final_glb, final_ply, final_preview, final_manifest]
missing = [path for path in required_outputs if not path.exists()]

if missing:
    print("Missing outputs:")
    for path in missing:
        print("  ", path)
else:
    print("All required Stage 2 outputs exist.")
    final_mesh = ensure_mesh(trimesh.load(final_glb, force="mesh"))
    print(json.dumps(mesh_health(final_mesh), indent=2))

print("\nOpen this preview before starting alignment:")
print(final_preview)


## Required Stage 2 outputs

```text
data/hunyuan_outputs/runs/run_001/candidates/seed_20260806/intact_prior.glb
data/hunyuan_outputs/runs/run_001/candidates/seed_20260806/intact_prior.ply
data/hunyuan_outputs/runs/run_001/candidates/seed_20260806/intact_prior_preview.html
data/hunyuan_outputs/runs/run_001/candidates/seed_20260806/prior_manifest.json
```

Only `intact_prior.glb` moves into alignment. Candidate folders remain as provenance and make the manual choice reproducible.
